# Notebook 01 Data Wrangling & Preprocessing: Student Burnout Dataset
**Proyek Analisis Data: HAPI (Human Activity Pattern Intelligence)**

### Profil Proyek & Tim
- **Tema Capstone:** Healthy Lives & Well-being
- **Target User:** Mahasiswa (fokus pada aspek akademik & lingkungan belajar)
- **Tujuan Proyek:** Pengembangan *platform* Webapp untuk *mood management & tracker* serta diagnosis mandiri tingkat kelelahan (*fatigue*).
- **Tim DS:** Greycia Febrina Michelle (CDCC700D6X2644) & Khazel Hayfa Yosmi (CDCC308D6X0629)

### Tujuan Notebook
Melakukan proses Data Wrangling secara *end-to-end* (Gathering → Assessing → Cleaning) pada **Student Burnout Dataset**. Langkah ini bertujuan untuk membersihkan data dari *missing values*, *outliers*, atau ketidakkonsistenan format, sehingga menghasilkan dataset yang bersih dan siap digunakan untuk analisis lanjutan maupun pemodelan.

### Peran Dataset
Dataset yang digunakan dalam tahapan ini bersumber dari Kaggle: [Student Burnout Dataset](https://www.kaggle.com/datasets/cereycie/student-burnout-dataset).

Dataset ini menjadi pondasi utama dari riset proyek HAPI karena menyediakan *ground truth* atau label mengenai tingkat *burnout* dan stres mahasiswa menggunakan standar psikometrik (seperti instrumen **MBI-SS: Maslach Burnout Inventory-Student Survey** yang divalidasi oleh Schaufeli et al., 2002). 

Data yang telah dibersihkan nantinya akan digunakan untuk:
1. Membangun model prediksi dasar (*baseline model*) untuk memetakan standar klinis stres pada populasi mahasiswa.
2. Memproses input dari fitur **Kuis MBI Baseline** pada aplikasi HAPI.

### Kolom Target
- `fatigue_score`: skor kontinu (0.0–6.0) yang dihitung berdasarkan formula standar MBI-SS.
- `risk_level`: kategori tingkat risiko (Low / Medium / High) yang diturunkan dari `fatigue_score` sebagai target utama model.

### Referensi Standar Psikometrik
- Schaufeli, W. B., Martinez, I. M., Pinto, A. M., Salanova, M., & Bakker, A. B. (2002). Burnout and engagement in university students: A cross-national study. Journal of cross-cultural psychology, 33(5), 464-481.
- Portoghese I, Leiter MP, Maslach C, Galletta M, Porru F, D’Aloja E, Finco G and Campagna M (2018) Measuring Burnout Among University Students: Factorial Validity, Invariance, and Latent Profiles of the Italian Version of the Maslach Burnout Inventory Student Survey (MBI-SS). Front. Psychol. 9:2105. doi: 10.3389/fpsyg.2018.02105

## Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

ROOT = Path.cwd()
for _ in range(5):
    if (ROOT / 'data').exists():
        break
    ROOT = ROOT.parent

RAW_PATH   = ROOT / 'data' / 'raw' / 'model_ready' / 'mbi_questions_dirty.csv'
CLEAN_PATH = ROOT / 'data' / 'clean' / 'model_ready' / 'mbi_questions_clean.csv'
PREP_PATH  = ROOT / 'data' / 'preprocessed' / 'mbi_questions_preprocessed.csv'

CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
PREP_PATH.parent.mkdir(parents=True, exist_ok=True)

COLS_EX    = [f'EX{i}' for i in range(1, 6)]
COLS_CY    = [f'CY{i}' for i in range(1, 5)]
COLS_EF    = [f'EF{i}' for i in range(1, 7)]
COLS_ITEMS = COLS_EX + COLS_CY + COLS_EF

print(f'ROOT     : {ROOT}')
print(f'File ada : {RAW_PATH.exists()}')
print('Setup selesai.')

ROOT     : d:\Proyek_Analisis_Burnout
File ada : True
Setup selesai.


**Dokumentasi Setup Data: MBI Questions**  
**Proyek HAPI (Human Activity Pattern Intelligence)**

Script ini berfungsi sebagai inisialisasi lingkungan kerja untuk pemrosesan data kuesioner MBI. Fokus utamanya adalah memetakan lokasi direktori proyek secara dinamis dan menyiapkan skema kolom sesuai standar instrumen MBI-SS.

**Operasi Utama**

**Path Mapping**  
Menggunakan `pathlib` untuk mendeteksi root direktori proyek secara otomatis. Ini memastikan script berjalan konsisten baik di lingkungan lokal maupun cloud.

**Directory Setup**  
Membuat folder penyimpanan secara otomatis jika belum tersedia. Kita membagi alur data menjadi dua tahap utama:

- `RAW_PATH`: Memuat data mentah (`mbi_questions_dirty.csv`).
- `CLEAN_PATH` & `PREP_PATH`: Menampung hasil pembersihan dan feature engineering sebelum masuk ke model.

**Schema Definition**  
Mengelompokkan 15 item kuesioner ke dalam dimensi burnout akademik:

- **EX (Exhaustion):** 5 item.
- **CY (Cynicism):** 4 item.
- **EF (Efficacy):** 6 item.

**Alur Kerja Data**

```text
mbi_questions_dirty.csv
          │
          ▼
      RAW_PATH
          │
          ▼
 Data Cleaning
          │
          ▼
     CLEAN_PATH
          │
          ▼
Feature Engineering
          │
          ▼
      PREP_PATH
          │
          ▼
     AI Model
```

**Catatan Teknis**

Kita mengatur konfigurasi `pandas` agar menampilkan seluruh kolom secara penuh dan membatasi presisi angka floating point menjadi 4 digit. Langkah ini krusial agar pengecekan data saat tahap Assessing lebih akurat sebelum data dikirim ke tim AI Engineer.

## Gathering Data

In [2]:
df_raw = pd.read_csv(RAW_PATH)
print(f'Dataset berhasil dimuat: {df_raw.shape[0]:,} baris, {df_raw.shape[1]} kolom')
print(f'Kolom: {df_raw.columns.tolist()}')
df_raw.head()

Dataset berhasil dimuat: 10,915 baris, 19 kolom
Kolom: ['student_id', 'EX1', 'EX2', 'EX3', 'EX4', 'EX5', 'CY1', 'CY2', 'CY3', 'CY4', 'EF1', 'EF2', 'EF3', 'EF4', 'EF5', 'EF6', 'fatigue_score', 'risk_level', 'row_status']


,student_id,EX1,EX2,EX3,EX4,EX5,CY1,CY2,CY3,CY4,EF1,EF2,EF3,EF4,EF5,EF6,fatigue_score,risk_level,row_status
0,MHS_03305,4.0000,2,4,2,4,4,3,3.0000,3,5,5,4,1,4,2,2.9800,Medium,clean
1,MHS_02871,1.0000,1,3,1,2,3,2,3.0000,2,4,4,4,4,3,2,2.2000,Medium,clean
2,MHS_09303,3.0000,1,4,2,2,0,3,3.0000,3,3,4,3,3,6,3,2.3300,Medium,clean
3,MHS_03826,4.0000,5,6,5,5,6,6,5.0000,5,0,1,1,0,2,2,5.1700,High,clean
4,MHS_03490,2.0000,4,4,4,1,0,2,2.0000,3,4,4,4,4,1,3,2.4700,Medium,clean


**Pemuatan Dataset (Loading Data)**

**Penjelasan Singkat**

- `pd.read_csv(RAW_PATH)`: Membaca file data.
- `df_raw.shape`: Menampilkan dimensi data, yaitu jumlah baris dan kolom.
- `df_raw.columns.tolist()`: Menampilkan daftar nama kolom agar kita tahu struktur atribut yang tersedia.
- `df_raw.head()`: Menampilkan lima baris pertama untuk melihat contoh isi data.

**Ringkasan Data**

Hasil pemuatan menunjukkan dataset terdiri dari **10.915 baris** dan **19 kolom**. Struktur kolom mencakup identitas mahasiswa, berbagai fitur numerik (`EX`, `CY`, `EF`), skor kelelahan, tingkat risiko, dan status baris. Berdasarkan cuplikan tersebut, data terlihat sudah dalam format yang rapi dan siap untuk diproses ke tahap pembersihan atau analisis lebih lanjut.

## Assessing Data

In [3]:
print('Shape')
print(f'Baris: {df_raw.shape[0]:,} | Kolom: {df_raw.shape[1]}')

Shape
Baris: 10,915 | Kolom: 19


In [4]:
print('Tipe Data')
print(df_raw.dtypes)

Tipe Data
student_id        object
EX1              float64
EX2                int64
EX3                int64
EX4                int64
EX5                int64
CY1                int64
CY2                int64
CY3              float64
CY4                int64
EF1                int64
EF2                int64
EF3                int64
EF4                int64
EF5                int64
EF6                int64
fatigue_score    float64
risk_level        object
row_status        object
dtype: object


In [5]:
print('Missing Values')
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'jumlah_missing': missing, 'persen_missing': missing_pct})
print(missing_df[missing_df['jumlah_missing'] > 0])

Missing Values
               jumlah_missing  persen_missing
EX1                       150          1.3700
CY3                       120          1.1000
fatigue_score              50          0.4600


In [6]:
print('Duplikat')
n_dup = df_raw.duplicated(subset=COLS_ITEMS).sum()
print(f'Baris duplikat (berdasarkan 15 item): {n_dup}')

Duplikat
Baris duplikat (berdasarkan 15 item): 42


In [7]:
print('Statistik Deskriptif (Item Kuesioner)')
print(df_raw[COLS_ITEMS].describe().round(2))

Statistik Deskriptif (Item Kuesioner)
             EX1        EX2        EX3        EX4        EX5        CY1  \
count 10765.0000 10915.0000 10915.0000 10915.0000 10915.0000 10915.0000   
mean      2.7500     3.0200     2.7500     2.7700     2.7600     2.6100   
std       1.7300     5.3200     1.7100     1.7200     1.7100     1.6900   
min       0.0000     0.0000     0.0000     0.0000     0.0000     0.0000   
25%       1.0000     1.0000     1.0000     1.0000     1.0000     1.0000   
50%       3.0000     3.0000     3.0000     3.0000     3.0000     2.0000   
75%       4.0000     4.0000     4.0000     4.0000     4.0000     4.0000   
max       6.0000    99.0000     6.0000     6.0000     6.0000     6.0000   

             CY2        CY3        CY4        EF1        EF2        EF3  \
count 10915.0000 10795.0000 10915.0000 10915.0000 10915.0000 10915.0000   
mean      2.6000     2.6100     2.5900     3.4400     3.4800     3.4600   
std       1.6800     1.6800     1.6800     1.6900     1.6300 

In [8]:
print('Validitas Nilai (Skala Valid: 0–6)')
for col in COLS_ITEMS:
    invalid = df_raw[col].dropna()
    out_of_range = ((invalid < 0) | (invalid > 6)).sum()
    if out_of_range > 0:
        print(f'  {col}: {out_of_range} nilai di luar rentang 0–6 '
              f'(min={invalid.min()}, max={invalid.max()})')
print('Selesai cek validitas.')

Validitas Nilai (Skala Valid: 0–6)
  EX2: 30 nilai di luar rentang 0–6 (min=0, max=99)
  EF1: 25 nilai di luar rentang 0–6 (min=-5, max=6)
Selesai cek validitas.


In [9]:
print('Inkonsistensi Label Risk Level')
print(df_raw['risk_level'].value_counts(dropna=False))

Inkonsistensi Label Risk Level
risk_level
Medium     4557
Low        3539
High       2319
Med-ium     500
Name: count, dtype: int64


In [10]:
print('Distribusi Row Status')
print(df_raw['row_status'].value_counts())

Distribusi Row Status
row_status
clean                          10000
dirty_typo_risk_level            500
dirty_missing_EX1                150
dirty_missing_CY3                120
dirty_missing_fatigue_score       50
dirty_duplicate                   40
dirty_outlier_EX2_high            30
dirty_outlier_EF1_low             25
Name: count, dtype: int64


**Profil Dataset**

Kami mengolah **10.915 entri** dengan **19 kolom** yang mencakup ID mahasiswa, variabel psikometrik, hingga status risiko. Secara struktural, sebagian besar tipe data sudah sesuai, namun kami menemukan beberapa anomali yang perlu segera diperbaiki.

**Temuan Kritis**

Kami mengidentifikasi beberapa titik masalah yang akan mengganggu hasil analisis jika dibiarkan:

- **Data Kosong:** Ada celah pada kolom `EX1`, `CY3`, dan `fatigue_score`. Kami perlu memutuskan apakah akan melakukan imputasi atau menghapus baris tersebut.

- **Anomali Rentang:** Kami menemukan nilai di luar batas logika (0 sampai 6). Khususnya pada kolom `EX2` dengan nilai maksimum `99` dan `EF1` yang mencatat angka negatif.

- **Inkonsistensi Kategori:** Label `risk_level` memiliki entri `Med-ium` yang menyimpang dari kategori utama. Ini kemungkinan besar adalah kesalahan ketik yang harus diseragamkan.

- **Duplikasi:** Terdapat **42 baris duplikat** yang terdeteksi berdasarkan **15 item kuesioner**.

**Ringkasan Hasil Assessing**

| Masalah | Detail | Tindakan |
|----------|----------|----------|
| Missing values | EX1 (~150 baris), CY3 (~120 baris), fatigue_score (~50 baris) | Hapus baris (missing di item kuesioner tidak bisa diimputasi — mengubah makna psikometri) |
| Outlier/invalid | EX2 = 99, EF1 = -5 (di luar skala 0–6) | Hapus baris |
| Duplikat | ~40 baris duplikat berdasarkan 15 item | Hapus baris |
| Typo risk_level | 'low', 'HIGH_RISK', 'Med-ium' | Hapus baris |
| Kolom tidak diperlukan | `row_status`, `student_id` | Drop sebelum export ke AI Engineer |

## Cleaning Data

In [11]:
df = df_raw.copy()
before = len(df)

df = df[df['row_status'] == 'clean'].copy()
print(f'Drop Baris Kotor: {before - len(df)} baris dihapus → sisa {len(df):,}')

Drop Baris Kotor: 915 baris dihapus → sisa 10,000


In [12]:
df = df.drop(columns=['row_status'])
print(f'Drop Kolom Row Status → kolom tersisa: {df.columns.tolist()}')

Drop Kolom Row Status → kolom tersisa: ['student_id', 'EX1', 'EX2', 'EX3', 'EX4', 'EX5', 'CY1', 'CY2', 'CY3', 'CY4', 'EF1', 'EF2', 'EF3', 'EF4', 'EF5', 'EF6', 'fatigue_score', 'risk_level']


In [13]:
assert df.isnull().sum().sum() == 0, 'Masih ada missing values!'
print(f'Missing Values: {df.isnull().sum().sum()} ✓')

Missing Values: 0 ✓


In [14]:
for col in COLS_ITEMS:
    assert ((df[col] >= 0) & (df[col] <= 6)).all(), f'{col} masih punya nilai di luar rentang!'
print('Semua Item Dalam Rentang 0–6 ✓')

Semua Item Dalam Rentang 0–6 ✓


In [15]:
before = len(df)
df = df.drop_duplicates(subset=COLS_ITEMS)
n_dup_removed = before - len(df)
print(f'Duplikat Dihapus: {n_dup_removed} baris → sisa {len(df):,}')

Duplikat Dihapus: 2 baris → sisa 9,998


In [16]:
valid_levels = {'Low', 'Medium', 'High'}
actual_levels = set(df['risk_level'].unique())
assert actual_levels == valid_levels, f'risk_level tidak valid: {actual_levels}'
print(f'Risk Level Valid: {sorted(actual_levels)} ✓')

Risk Level Valid: ['High', 'Low', 'Medium'] ✓


In [17]:
df[COLS_ITEMS] = df[COLS_ITEMS].astype(int)
print('Tipe Data Item Dikastkan ke Int ✓')

print(f'\nShape Akhir Cleaned: {df.shape}')
df.head(3)

Tipe Data Item Dikastkan ke Int ✓

Shape Akhir Cleaned: (9998, 18)


,student_id,EX1,EX2,EX3,EX4,EX5,CY1,CY2,CY3,CY4,EF1,EF2,EF3,EF4,EF5,EF6,fatigue_score,risk_level
0,MHS_03305,4,2,4,2,4,4,3,3,3,5,5,4,1,4,2,2.9800,Medium
1,MHS_02871,1,1,3,1,2,3,2,3,2,4,4,4,4,3,2,2.2000,Medium
2,MHS_09303,3,1,4,2,2,0,3,3,3,3,4,3,3,6,3,2.3300,Medium


In [18]:
df.to_csv(CLEAN_PATH, index=False)
print(f'Dataset Cleaned Disimpan Ke: {CLEAN_PATH}')

Dataset Cleaned Disimpan Ke: d:\Proyek_Analisis_Burnout\data\clean\model_ready\mbi_questions_clean.csv


**Pembersihan Data untuk Dataset Burnout**

Kami melakukan serangkaian langkah pembersihan data guna memastikan kualitas input sebelum masuk ke tahap pemodelan. Berikut proses yang kami jalankan.

**Langkah-Langkah Teknis**

- **Penyaringan Data Mentah:** Kami membuang **915 baris** yang ditandai sebagai kotor (`status` bukan `'clean'`) agar data yang diproses hanya data yang valid.

- **Pemangkasan Kolom:** Kolom `row_status` dihapus karena tidak lagi diperlukan setelah proses filter.

- **Verifikasi Integritas:** Kami memastikan tidak ada *missing values* tersisa dan melakukan pengecekan rentang nilai untuk setiap item kuesioner agar berada di skala **0 sampai 6**.

- **Penanganan Duplikat:** Ditemukan **2 baris data duplikat** berdasarkan kolom item kuesioner, yang kemudian kami hapus.

- **Validasi Label:** Kami memastikan kategori `risk_level` hanya berisi label yang benar yaitu `Low`, `Medium`, dan `High`.

- **Penyesuaian Tipe Data:** Seluruh kolom item kuesioner dikonversi menjadi integer agar sesuai dengan format yang dibutuhkan model.

**Hasil Akhir**

Setelah semua proses tadi, kami mendapatkan dataset bersih dengan total **9.998 baris** dan **18 kolom** siap pakai. Data final ini kami simpan ke dalam file `mbi_questions_clean.csv` di direktori yang sudah ditentukan.

## Preprocessing & Feature Engineering

In [19]:
df_prep = pd.read_csv(CLEAN_PATH)

mean_ex     = df_prep[COLS_EX].mean(axis=1)
mean_cy     = df_prep[COLS_CY].mean(axis=1)
mean_ef_inv = (6 - df_prep[COLS_EF]).mean(axis=1)
fatigue_recalc = ((mean_ex + mean_cy + mean_ef_inv) / 3).round(2)

match_rate = (fatigue_recalc == df_prep['fatigue_score']).mean()
print(f'Formula Match Rate: {match_rate:.2%}')
print(f'Fatigue Score Range: {df_prep["fatigue_score"].min():.2f} – {df_prep["fatigue_score"].max():.2f}')
print('Theoretical Range: 0.00 – 6.00')

assert match_rate == 1.0, 'PERINGATAN: formula tidak konsisten, periksa kolom target!'

Formula Match Rate: 100.00%
Fatigue Score Range: 0.29 – 5.48
Theoretical Range: 0.00 – 6.00


### Feature Engineering

In [20]:
df_prep['dim_exhaustion']  = df_prep[COLS_EX].mean(axis=1).round(4)
df_prep['dim_cynicism']    = df_prep[COLS_CY].mean(axis=1).round(4)
df_prep['dim_efficacy_inv'] = (6 - df_prep[COLS_EF]).mean(axis=1).round(4)

print('Fitur Dimensi MBI Berhasil Dibuat')
print(df_prep[['dim_exhaustion', 'dim_cynicism', 'dim_efficacy_inv', 'fatigue_score']].describe().round(3))

Fitur Dimensi MBI Berhasil Dibuat
       dim_exhaustion  dim_cynicism  dim_efficacy_inv  fatigue_score
count       9998.0000     9998.0000         9998.0000      9998.0000
mean           2.7440        2.5980            2.5020         2.6140
std            1.5740        1.5450            1.4610         1.4900
min            0.0000        0.0000            0.0000         0.2900
25%            1.2000        1.2500            1.0000         1.0900
50%            2.6000        2.5000            2.5000         2.5900
75%            4.0000        3.7500            3.6670         3.9300
max            6.0000        6.0000            5.8330         5.4800


### Leakage Check

In [21]:
FEATURE_COLS = COLS_ITEMS + ['dim_exhaustion', 'dim_cynicism', 'dim_efficacy_inv']
TARGET_COLS  = ['fatigue_score', 'risk_level']

leakage = set(FEATURE_COLS) & set(TARGET_COLS)
assert len(leakage) == 0, f'DATA LEAKAGE TERDETEKSI: {leakage}'

print('Leakage Check: BERSIH ✓')
print(f'Feature Columns ({len(FEATURE_COLS)}): {FEATURE_COLS}')
print(f'Target Columns ({len(TARGET_COLS)}): {TARGET_COLS}')

Leakage Check: BERSIH ✓
Feature Columns (18): ['EX1', 'EX2', 'EX3', 'EX4', 'EX5', 'CY1', 'CY2', 'CY3', 'CY4', 'EF1', 'EF2', 'EF3', 'EF4', 'EF5', 'EF6', 'dim_exhaustion', 'dim_cynicism', 'dim_efficacy_inv']
Target Columns (2): ['fatigue_score', 'risk_level']


### Encoding dan Finalisasi

In [22]:
risk_map = {'Low': 0, 'Medium': 1, 'High': 2}
df_prep['risk_level_encoded'] = df_prep['risk_level'].map(risk_map)

print('Encoding Risk Level')
print(df_prep[['risk_level', 'risk_level_encoded']].value_counts().sort_index())

Encoding Risk Level
risk_level  risk_level_encoded
High        2                     2312
Low         0                     3517
Medium      1                     4169
Name: count, dtype: int64


In [23]:
FINAL_COLS = (
    ['student_id'] +
    COLS_ITEMS +
    ['dim_exhaustion', 'dim_cynicism', 'dim_efficacy_inv'] +
    ['fatigue_score', 'risk_level', 'risk_level_encoded']
)

df_final = df_prep[FINAL_COLS].copy()

print(f'Shape Final: {df_final.shape}')
print(f'Missing Values: {df_final.isnull().sum().sum()}')
df_final.head(3)

Shape Final: (9998, 22)
Missing Values: 0


,student_id,EX1,EX2,EX3,EX4,EX5,CY1,CY2,CY3,CY4,EF1,EF2,EF3,EF4,EF5,EF6,dim_exhaustion,dim_cynicism,dim_efficacy_inv,fatigue_score,risk_level,risk_level_encoded
0,MHS_03305,4,2,4,2,4,4,3,3,3,5,5,4,1,4,2,3.2000,3.2500,2.5000,2.9800,Medium,1
1,MHS_02871,1,1,3,1,2,3,2,3,2,4,4,4,4,3,2,1.6000,2.5000,2.5000,2.2000,Medium,1
2,MHS_09303,3,1,4,2,2,0,3,3,3,3,4,3,3,6,3,2.4000,2.2500,2.3333,2.3300,Medium,1


In [24]:
df_final.to_csv(PREP_PATH, index=False)

print(f'Dataset Model-Ready Disimpan Ke: {PREP_PATH}')

print('\nRingkasan Final')
print(f'Total Baris: {len(df_final):,}')
print(f'Total Fitur (X): {len(COLS_ITEMS) + 3} (15 item + 3 dimensi)')
print('Kolom Target (Y): fatigue_score (regresi) | risk_level_encoded (klasifikasi)')
print('Missing Values: 0')
print('Leakage: BERSIH')

print('\nDistribusi Risk Level')
print(df_final['risk_level'].value_counts())

Dataset Model-Ready Disimpan Ke: d:\Proyek_Analisis_Burnout\data\preprocessed\mbi_questions_preprocessed.csv

Ringkasan Final
Total Baris: 9,998
Total Fitur (X): 18 (15 item + 3 dimensi)
Kolom Target (Y): fatigue_score (regresi) | risk_level_encoded (klasifikasi)
Missing Values: 0
Leakage: BERSIH

Distribusi Risk Level
risk_level
Medium    4169
Low       3517
High      2312
Name: count, dtype: int64


**Rekayasa Fitur untuk Pemodelan Burnout**

Kami telah menyelesaikan tahap persiapan fitur agar data siap digunakan oleh model machine learning. Fokus utama kami di sini adalah memastikan validitas perhitungan dan integritas data agar tidak terjadi kebocoran informasi.

**Verifikasi dan Transformasi Data**

- **Audit Formula:** Kami menguji ulang konsistensi `fatigue_score` terhadap standar Schaufeli. Hasilnya **100% akurat**. Ini memberikan keyakinan penuh bahwa kolom target kita valid.

- **Ekstraksi Dimensi:** Dari **15 item kuesioner mentah**, kami merangkumnya ke dalam **3 fitur dimensi utama** yaitu `exhaustion`, `cynicism`, dan `efficacy`. Fitur ini membantu model memahami pola psikometri di balik skor mentah.

- **Encoding Target:** Label kategori `risk_level` kami konversi ke format numerik (`0`, `1`, `2`) supaya bisa diproses oleh algoritma klasifikasi.

**Jaminan Kualitas**

Kami menerapkan beberapa prosedur untuk mencegah kegagalan model di masa depan:

- **Cek Kebocoran Data:** Kami memastikan kolom target (`fatigue_score` dan `risk_level`) tidak tercampur ke dalam fitur input. Ini langkah vital supaya model tidak curang saat proses training.

- **Struktur Final:** Dataset kini memiliki **22 kolom** yang mencakup ID, skor mentah, fitur dimensi, serta target yang sudah disiapkan.

- **Penyimpanan:** File hasil akhir sudah tersimpan di `mbi_questions_preprocessed.csv`.

**Ringkasan**

Ringkasnya, kami sudah memiliki **18 fitur input** yang bersih dan siap diuji ke berbagai model regresi maupun klasifikasi. Langkah selanjutnya adalah menentukan algoritma mana yang paling cocok dengan distribusi data ini.